In [ ]:
OPENROUTER_API_KEY = ''

In [ ]:
Z_SHOT_SYSTEM_PROMPT = """
Suppose you are a Senior AI Engineer tasked with building a LangGraph agent. You will be provided the <code intent> that describes what the agent is supposed to do.
Your code should have the following structures:
Setup (First line): !pip install -U [all requirements]
API Keys: Use 'userdata.get('OPENAI_API_KEY')' for the agent.
Should include relevant tools multiple tools and agents if needed.
NO prose. NO comments.

Your code will also be tested on already build test cases. For that, your code should be formatted in a way to be tested. It should take a string as input and return a string as output.


Now, write the code and return the code in this format:

<code>
{your_code_here}
</code>
"""

O_SHOT_SYSTEM_PROMPT = """
Suppose you are a Senior AI Engineer tasked with building a LangGraph agent. You will be provided the <code intent> that describes what the agent is supposed to do.
Your code should have the following structures:
Setup (First line): !pip install -U [all requirements]
API Keys: Use 'userdata.get('OPENAI_API_KEY')' for the agent.
Should include relevant tools multiple tools and agents if needed.
NO prose. NO comments.

Your code will also be tested on already build test cases. For that, your code should be formatted in a way to be tested. It should take a string as input and return a string as output.

Example (One-Shot):
Input Intent: Create a weather assistant.
Output:
!pip install -U langgraph langchain_openai
import os
from google.colab import userdata
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

def get_weather(location: str):
    return f'The weather in {location} is 22°C and sunny.'

tools = [get_weather]
model = ChatOpenAI(model='gpt-4-turbo').bind_tools(tools)

class State(TypedDict):
    messages: Annotated[list, lambda x, y: x + y]

def call_model(state: State):
    return {'messages': [model.invoke(state['messages'])]}

builder = StateGraph(State)
builder.add_node('agent', call_model)
builder.add_node('tools', ToolNode(tools))
builder.set_entry_point('agent')
builder.add_conditional_edges('agent', lambda x: 'tools' if x['messages'][-1].tool_calls else END)
builder.add_edge('tools', 'agent')
app = builder.compile()

def run_test(query: str):
    inputs = {'messages': [('user', query)]}
    final_state = app.invoke(inputs)
    return final_state['messages'][-1].content

output = run_test('What is the weather in San Francisco?')
print(output)
"""

In [ ]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Configuration ---
FILE_PATH = "Dataset.xlsx"
OUTPUT_FILE = "processed_results_gpt.xlsx"
MAX_WORKERS = 5  # Number of parallel API calls. Adjust based on your rate limits.
MODELS = {
    "gpt": "openai/gpt-5.2",
    "claude": "anthropic/claude-sonnet-4.6",
    "deepseek": "deepseek/deepseek-v3.2"
}
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "http://localhost:3000",
        "X-Title": "Local Research Script",
    }
)

def get_ai_response(model, prompt, system_prompt):
    if pd.isna(prompt) or str(prompt).strip() == "":
        return "Skipped: Empty Prompt"
    try:
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": str(system_prompt)},
                {"role": "user", "content": str(prompt)}
            ]
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

def process_row(index, row, input_col):
    """
    Function to process a single row.
    Returns the index and the results so we can update the dataframe.
    """
    z_out = ""
    o_out = ""

    prompt_text = row[input_col]

    # Process Z-shot
    z_out = get_ai_response(MODELS["gpt"], prompt_text, Z_SHOT_SYSTEM_PROMPT)
    # Process One-shot
    o_out = get_ai_response(MODELS["gpt"], prompt_text, O_SHOT_SYSTEM_PROMPT)

    return index, z_out, o_out

def process_sheets():
    try:
        if not os.path.exists(FILE_PATH):
            print(f"File not found: {FILE_PATH}")
            return

        all_sheets = pd.read_excel(FILE_PATH, sheet_name=None)
        target_sheets = ["GitHub", "Literature"]
        working_sheets = {}

        for name in target_sheets:
            if name not in all_sheets:
                print(f"Skipping {name}: Not found.")
                continue

            df = all_sheets[name]
            if "Z-shot Output" not in df.columns: df["Z-shot Output"] = ""
            if "One-shot Output" not in df.columns: df["One-shot Output"] = ""
            working_sheets[name] = df

        for name, df in working_sheets.items():
            print(f"\n--- Parallel Processing: {name} ---")
            input_col = "Agent Intent"

            # Filter rows that actually need processing
            to_process = df[
                (df["Z-shot Output"].astype(str).str.strip() == "") |
                (df["One-shot Output"].astype(str).str.strip() == "")
            ]

            if to_process.empty:
                print(f"All rows in {name} are already processed.")
                continue

            # Use ThreadPoolExecutor for parallel requests
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = {
                    executor.submit(process_row, idx, row, input_col): idx
                    for idx, row in to_process.iterrows()
                }

                count = 0
                for future in tqdm(as_completed(futures), total=len(futures), desc=f"Threads ({name})"):
                    idx, z_res, o_res = future.result()
                    df.at[idx, "Z-shot Output"] = z_res
                    df.at[idx, "One-shot Output"] = o_res

                    # Batch save every 10 rows to avoid file corruption/slowdown
                    count += 1
                    if count % 10 == 0:
                        with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
                            for s_name, s_df in working_sheets.items():
                                s_df.to_excel(writer, sheet_name=s_name, index=False)

        # Final save
        with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
            for s_name, s_df in working_sheets.items():
                s_df.to_excel(writer, sheet_name=s_name, index=False)

        print(f"\nSuccess! Results saved to {OUTPUT_FILE}")

    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == "__main__":
    process_sheets()


--- Parallel Processing: GitHub ---


Threads (GitHub): 100%|██████████| 62/62 [22:45<00:00, 22.02s/it]



--- Parallel Processing: Literature ---


Threads (Literature): 100%|██████████| 29/29 [12:49<00:00, 26.52s/it]


Success! Results saved to processed_results_gpt.xlsx
